<div dir="rtl">

# 🧠 05 - End-to-End Complete Embeddings & Semantic Search Pipeline (من الصفر للاحتراف)

## ما هي التضمينات النصية (Text Embeddings)؟
- تحويل النصوص والكلمات إلى متجهات عددية (**Dense Vectors**) في فضاء دلالي متعدد الأبعاد.
- النصوص التي تتشابه في **المعنى والسياق** تصبح متقاربة جداً في الفضاء المتجهي، مما يتيح **البحث الدلالي (Semantic Search)** حتى لو اختلفت الكلمات الحرفية.

---

### 🎯 ما سنتعلمه ونطبقه في هذا الكراس:
1. **استيعاب وتقسيم المستند الأصلي** إلى قطع نصية متوازنة.
2. **استخراج التضمينات محلياً 100%** عبر `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`).
3. **تجربة التضمينات فائقة السرعة** عبر محرك `FastEmbed` المبني على ONNX.
4. **بناء نظام كاش محلي (Cache-Backed Embeddings)** لتسريع الأداء وتفادي إعادة حساب نفس النصوص.
5. **تطبيق البحث الدلالي وحساب Cosine Similarity يدوياً** باستخدام `numpy`.
6. **مقارنة استعلامات بحثية متنوعة** وتحليل المسافات الدلالية.

</div>


<div dir="rtl">

### 1️⃣ تجهيز البيئة والمستندات التجريبية

</div>


In [ ]:
import os
import time
import numpy as np
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain.embeddings import CacheBackedEmbeddings
from langchain.storage import LocalFileStore

load_dotenv(find_dotenv())

# قراءة الملف التجريبي
DATA_DIR = Path("data") if Path("data").exists() else Path("../../data") if Path("../../data").exists() else Path("../data")
sample_file = DATA_DIR / "sample.txt"

loader = TextLoader(str(sample_file), encoding="utf-8")
raw_docs = loader.load()

# تقسيم المستند إلى قطع مركزة
splitter = RecursiveCharacterTextSplitter(chunk_size=350, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)

print(f"✅ تم تحميل وتقسيم المستند إلى {len(chunks)} قطعة جاهزة للتضمين!")


<div dir="rtl">

### 2️⃣ تهيئة نماذج التضمين المحلية واستخراج المتجهات
نستخدم نموذج `sentence-transformers/all-MiniLM-L6-v2` الذي يُنتج متجهات ذات 384 بعداً.

</div>


In [ ]:
hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

chunk_texts = [c.page_content for c in chunks]

start_time = time.time()
vectors = hf_embeddings.embed_documents(chunk_texts)
elapsed = time.time() - start_time

print(f"✅ تم تضمين {len(vectors)} قطعة بنجاح في {elapsed:.2f} ثانية!")
print(f"📊 أبعاد كل متجه: {len(vectors[0])} بعداً.")
print(f"عينة من أرقام المتجه الأول: {vectors[0][:5]}...")


<div dir="rtl">

### 3️⃣ بناء نظام كاش محلي للتضمينات (CacheBackedEmbeddings)
حفظ المتجهات في مجلد محلي على القرص (`LocalFileStore`) بحيث إذا مر نفس النص مستقبلاً يتم جلبه من الكاش في أجزاء من الميلي ثانية.

</div>


In [ ]:
cache_dir = DATA_DIR / "embedding_cache"
os.makedirs(cache_dir, exist_ok=True)

store = LocalFileStore(str(cache_dir))

# تغليف نموذج التضمين بالكاش
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=hf_embeddings,
    document_embedding_cache=store,
    namespace="minilm_v2"
)

# التجربة الأولى (حساب وحفظ في الكاش)
t1 = time.time()
_ = cached_embeddings.embed_documents(chunk_texts[:5])
time_first = time.time() - t1

# التجربة الثانية لنفس النصوص (جلب فوري من الكاش)
t2 = time.time()
_ = cached_embeddings.embed_documents(chunk_texts[:5])
time_cached = time.time() - t2

print(f"⏱️ زمن التضمين لأول مرة (حساب): {time_first:.4f} ثانية")
print(f"⚡ زمن الاسترجاع من الكاش: {time_cached:.6f} ثانية (تسريع مضاعف!)")


<div dir="rtl">

### 4️⃣ تطبيق البحث الدلالي وحساب الـ Cosine Similarity يدوياً
نقوم بتضمين استعلام المستخدم، ثم حساب معامل تشابه جيب التمام مع كافة متجهات المستندات.

</div>


In [ ]:
def cosine_similarity(v1, v2):
    dot = np.dot(v1, v2)
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    return dot / (norm1 * norm2)

query = "What is the role of RAG in reducing hallucinations?"
query_vector = hf_embeddings.embed_query(query)

# حساب التشابه لكل قطعة
similarities = []
for i, vec in enumerate(vectors):
    score = cosine_similarity(query_vector, vec)
    similarities.append((i, score))

# ترتيب النتائج من الأعلى للأقل شبهاً
similarities.sort(key=lambda x: x[1], reverse=True)

print(f"🔍 الاستعلام: '{query}'\n")
print("🏆 أفضل 3 نتائج متطابقة دلالياً:")
print("="*60)
for rank, (idx, score) in enumerate(similarities[:3], 1):
    print(f"[{rank}] درجة التطابق (Cosine Similarity): {score:.4f}")
    print(f"المحتوى: {chunks[idx].page_content.strip()[:150]}...")
    print("-" * 50)


<div dir="rtl">

### 5️⃣ تجربة استعلام باللغة العربية (Cross-Lingual Retrieval)
اختبار قدرة النموذج على التقاط المعنى المشترك بين اللغات.

</div>


In [ ]:
query_ar = "ما هي الفواصل المستخدمة في تقسيم النصوص؟"
query_ar_vector = hf_embeddings.embed_query(query_ar)

ar_similarities = [(i, cosine_similarity(query_ar_vector, vec)) for i, vec in enumerate(vectors)]
ar_similarities.sort(key=lambda x: x[1], reverse=True)

print(f"🔍 الاستعلام بالعربية: '{query_ar}'\n")
best_idx, best_score = ar_similarities[0]
print(f"🎯 أفضل نتيجة مطابقة (Score: {best_score:.4f}):")
print(chunks[best_idx].page_content.strip()[:200])
print("\n🎉 تم إنجاز مسار التضمينات والبحث الدلالي بنجاح تام!")
